# Delivery Cost Prediction


In [26]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "olist").exists())
sys.path.insert(0, str(ROOT / "src"))

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge

from olist import config as cfg, io_utils as io, modeling as ml, plots
from olist.features import PURCHASE_TIME_FEATURES

pd.set_option("display.max_columns", 120); pd.set_option("display.width", 220)
cfg.set_seed()
PROJECT, TARGET = "03_freight_cost_prediction", "order_freight_total"
paths = ml.get_paths(PROJECT)
df = io.load_ml("order_model_base")

## 1. Feature contract


In [27]:
FREIGHT_DERIVED = ["order_freight_total", "freight_ratio", "freight_per_kg"]
FEATURES = [c for c in PURCHASE_TIME_FEATURES if c in df.columns and c not in FREIGHT_DERIVED]

leakage

In [28]:
for c in FREIGHT_DERIVED:
    print(f"  - {c}")
print(f"\n{len(FEATURES)} features retained")
ml.leakage_audit(FEATURES, TARGET)

  - order_freight_total
  - freight_ratio
  - freight_per_kg

36 features retained


,feature,verdict,reason
0,order_item_count,allowed,known at purchase time
1,order_product_count,allowed,known at purchase time
2,order_seller_count,allowed,known at purchase time
3,order_gmv,allowed,known at purchase time
4,avg_item_price,allowed,known at purchase time
5,max_item_price,allowed,known at purchase time
6,is_multi_seller,allowed,known at purchase time
7,order_weight_g,allowed,known at purchase time
8,order_volume_cm3,allowed,known at purchase time
9,max_item_weight_g,allowed,known at purchase time


In [29]:
# Demonstrate what the shortcut would have bought.
leaky = FEATURES + ["freight_ratio"]
leak_data = ml.chronological_split(df, leaky, TARGET)
leak_pipe = ml.make_pipeline(Ridge(alpha=1.0), ml.to_sklearn_frame(leak_data.X_train))
leak_pipe.fit(ml.to_sklearn_frame(leak_data.X_train), leak_data.y_train)
leak_r2 = ml.regression_metrics(leak_data.y_test,
                                leak_pipe.predict(ml.to_sklearn_frame(leak_data.X_test)))["r2"]

In [30]:
data = ml.chronological_split(df, FEATURES, TARGET)

## 2. Evaluation setup


In [31]:
y_test, X_test = data.y_test, data.X_test
train = df[(df["split"] == "train") & df[TARGET].notna()]

In [32]:
mean_metrics = ml.regression_metrics(y_test, np.full(len(y_test), float(data.y_train.mean())))
rate_card = train.groupby("distance_bucket", observed=True).apply(
    lambda g: g[TARGET].sum() / max(g["order_weight_g"].sum() / 1000, 1e-9), include_groups=False
).rename("brl_per_kg")
global_rate = train[TARGET].sum() / max(train["order_weight_g"].sum() / 1000, 1e-9)

In [33]:
print("training-period rate card (BRL per kg):")
print(rate_card.round(3).to_string())
print(f"fallback rate: {global_rate:.3f} BRL/kg\n")

training-period rate card (BRL per kg):
distance_bucket
1000km+       13.932
200-500km      8.193
50-200km       5.928
500-1000km     9.987
<50km          5.727
fallback rate: 9.027 BRL/kg



In [34]:
card_pred = (
    X_test["distance_bucket"].map(rate_card).fillna(global_rate).to_numpy(dtype=float)
    * np.nan_to_num(X_test["order_weight_g"].to_numpy(dtype=float), nan=0.0) / 1000
)
card_metrics = ml.regression_metrics(y_test, card_pred)
pd.DataFrame({"baseline: train mean": mean_metrics,
              "INCUMBENT: rate card (BRL/kg by distance)": card_metrics}).T.round(3)

,mae,rmse,r2,mape_pct,median_abs_err
baseline: train mean,12.177,27.465,-0.006,57.697,7.351
INCUMBENT: rate card (BRL/kg by distance),16.338,25.306,0.146,69.956,12.233


## 3. Model candidates


In [35]:
candidates = {
    **ml.baseline_regressors(),
    "ridge": Ridge(alpha=10.0, random_state=cfg.RANDOM_STATE),
    "random_forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=20,
                                           random_state=cfg.RANDOM_STATE, n_jobs=cfg.N_JOBS),
    "hist_gbm": HistGradientBoostingRegressor(max_iter=350, learning_rate=0.06,
                                              random_state=cfg.RANDOM_STATE),
}

In [36]:
comparison, fitted = ml.compare_candidates(candidates, data, task="regression")
print(comparison.round(3).to_string(index=False))

          model    mae   rmse     r2  mape_pct  median_abs_err
          ridge  0.714  1.494  0.996     2.614           0.335
       hist_gbm  3.523  7.549  0.887    13.515           1.870
  random_forest  4.612 10.356  0.787    15.720           1.799
baseline_median 11.599 23.535 -0.100    41.886           5.840
  baseline_mean 12.349 22.493 -0.005    58.861           7.811


In [37]:
BEST_NAME = comparison.iloc[0]["model"]
best_pipe = fitted[BEST_NAME]
pred = best_pipe.predict(ml.to_sklearn_frame(X_test))
model_metrics = ml.regression_metrics(y_test, pred)

In [38]:
print(pd.DataFrame({"INCUMBENT: rate card": card_metrics,
                    f"model: {BEST_NAME}": model_metrics}).T.round(3).to_string())
print(f"\nMAE vs rate card: {1 - model_metrics['mae'] / card_metrics['mae']:+.1%}")

                         mae    rmse     r2  mape_pct  median_abs_err
INCUMBENT: rate card  16.338  25.306  0.146    69.956          12.233
model: ridge           0.663   2.033  0.994     2.309           0.304

MAE vs rate card: +95.9%


## 4. Revenue assurance read-out


In [39]:
actual_total, pred_total = float(y_test.sum()), float(pred.sum())
card_total = float(card_pred.sum())
print(f"actual freight in the test period : {actual_total:>12,.0f} BRL")
print(f"model total                       : {pred_total:>12,.0f} BRL  "
      f"({pred_total / actual_total - 1:+.2%})")
print(f"rate-card total                   : {card_total:>12,.0f} BRL  "
      f"({card_total / actual_total - 1:+.2%})")
print()
print(f"per-order MAE  model {model_metrics['mae']:.2f} BRL vs rate card {card_metrics['mae']:.2f} BRL")
print(f"per-order MAE as a share of mean freight: {model_metrics['mae'] / y_test.mean():.1%}")

actual freight in the test period :      311,864 BRL
model total                       :      308,457 BRL  (-1.09%)
rate-card total                   :      222,528 BRL  (-28.65%)

per-order MAE  model 0.66 BRL vs rate card 16.34 BRL
per-order MAE as a share of mean freight: 2.7%


In [40]:
importance = ml.importance_table(best_pipe, X_test, y_test, top_k=12, n_repeats=5,
                                scoring="neg_mean_absolute_error")
print(importance.round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(9, 6))
top = importance.head(10).iloc[::-1]
ax.barh(top["feature"], top["importance"], xerr=top["std"], color="#e9c46a")
ax.set(xlabel="Increase in MAE when shuffled (BRL)", title="What drives freight")
ax.grid(axis="x", alpha=0.3)
plots.stamp_and_save(fig, paths.figure("importance.png"),
    f"INSIGHT: '{importance.iloc[0]['feature']}' dominates ({importance.iloc[0]['importance']:.2f} BRL "
    f"of MAE), ahead of '{importance.iloc[1]['feature']}'. Freight is priced on physical properties and "
    f"distance, exactly as a carrier rate card would be - which is a good sanity check that the model "
    f"has learned the real mechanism rather than an artefact.")
plt.show()

         feature  importance   std
order_total_paid     138.961 0.443
       order_gmv     127.482 0.524
  avg_item_price       0.207 0.002
order_volume_cm3       0.178 0.002
  order_weight_g       0.142 0.002
order_item_count       0.136 0.002
  max_item_price       0.055 0.001
 same_state_flag       0.054 0.001
     distance_km       0.032 0.001
distance_km_mean       0.030 0.001
  Customer_State       0.023 0.000
 distance_bucket       0.014 0.001


In [41]:
saved = ml.save_artifacts(best_pipe, paths, target=TARGET, features=FEATURES,
    metrics=model_metrics,
    baseline_metrics={f"rate_card_{k}": v for k, v in card_metrics.items()},
    policy={"type": "cost_estimation", "per_order_mae_brl": model_metrics["mae"],
            "portfolio_error_pct": round((pred_total / actual_total - 1) * 100, 3),
            "quote_buffer_p90_brl": float(np.percentile(np.abs(pred - y_test.to_numpy(dtype=float)), 90))},
    model_name=BEST_NAME,
    notes="Freight-derived columns excluded from the matrix. Baseline is a training-period BRL/kg rate card.")
for k, v in saved.items(): print(f"{k:<10} -> {v.relative_to(ROOT)}")

model      -> models\03_freight_cost_prediction\best_model.joblib
metadata   -> models\03_freight_cost_prediction\model_metadata.json
